# 03 - Experiment Results Analysis & Visualization

This notebook provides a database-driven analysis dashboard for your evolutionary BBOB experiments. It reads directly from the SQLite database (`data/db.sqlite3`), loads the data into robust Pandas DataFrames using a MultiIndex hierarchy, performs statistical analysis, and generates interactive plots for comparison across problem IDs and LLMs.

In [1]:
# Setup paths and imports
import sys
from pathlib import Path

# Add project root src/ to sys.path regardless of directory name or launch folder
root_dir = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
src_path = str(root_dir / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import sqlite3
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from dotenv import load_dotenv
import os
load_dotenv()
db_url = os.environ.get("DATABASE_URL")
if db_url:
    if db_url.startswith("sqlite:///"):
        DB_PATH = Path(db_url[10:])
    else:
        DB_PATH = Path(db_url)
    if not DB_PATH.is_absolute():
        DB_PATH = root_dir / DB_PATH
else:
    DB_PATH = root_dir / 'data' / 'db.sqlite3'
print("Analysis dashboard environment initialized.")

Analysis dashboard environment initialized.


## 1. Load Data from SQLite
We load `experiments` and `iterations` from the SQLite database. Then, we join them to form a unified DataFrame. We apply a robust MultiIndex on `['problem_id', 'dim', 'mode', 'llm_name', 'experiment_id', 'iteration']` for hierarchical access.

In [2]:
def load_and_combine_data(db_path):
    if not db_path.exists():
        print(f"Database not found at {db_path}")
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
        
    conn = sqlite3.connect(db_path)
    try:
        # Load tables
        df_exp = pd.read_sql_query("SELECT * FROM experiments", conn)
        df_iter = pd.read_sql_query("SELECT * FROM iterations", conn)
        
        if df_exp.empty or df_iter.empty:
            return df_exp, df_iter, pd.DataFrame()
            
        # Combine into a single comprehensive DataFrame
        df_combined = pd.merge(
            df_iter, 
            df_exp, 
            left_on='experiment_id', 
            right_on='id', 
            suffixes=('_iter', '_exp')
        )
        
        # Create a MultiIndex hierarchy
        index_cols = ['problem_id', 'dim', 'mode', 'llm_name', 'experiment_id', 'iteration']
        df_combined.set_index(index_cols, inplace=True)
        df_combined.sort_index(inplace=True)
        
        return df_exp, df_iter, df_combined
    finally:
        conn.close()

df_exp, df_iter, df_combined = load_and_combine_data(DB_PATH)

print(f"Loaded {len(df_exp)} experiment runs.")
print(f"Loaded {len(df_iter)} iterations.")
print(f"Combined DataFrame shape: {df_combined.shape}")
if not df_combined.empty:
    display(df_combined.head(3))

Loaded 0 experiment runs.
Loaded 0 iterations.
Combined DataFrame shape: (0, 0)


## 2. Statistical Analysis & Experiment Report
Generate robust statistical groupings to understand metrics (Time, Errors) across different LLMs for the same problem, or vice versa.

In [3]:
if not df_exp.empty:
    print("=== Best Final Errors per Problem and LLM ===")
    
    # Using df_exp for final aggregate stats per run
    stats_df = df_exp.groupby(['problem_id', 'dim', 'mode', 'llm_name']).agg(
        Runs_Count=('id', 'count'),
        Min_Error=('best_final_error', 'min'),
        Mean_Error=('best_final_error', 'mean'),
        Median_Error=('best_final_error', 'median'),
        Std_Dev=('best_final_error', 'std')
    ).reset_index()
    
    display(stats_df.round(6))
    
    if not df_combined.empty:
        print("\n=== Execution Efficiency (Time Consumed) ===")
        # Group by problem_id and llm_name to compare times
        # We need to sum the runtime per experiment, then average across experiments
        # Reset index to easily group
        df_flat = df_combined.reset_index()
        
        # Total runtime per experiment
        exp_runtimes = df_flat.groupby(['problem_id', 'dim', 'mode', 'llm_name', 'experiment_id'])['runtime_seconds'].sum().reset_index()
        
        # Average runtime per LLM & Problem
        time_stats = exp_runtimes.groupby(['problem_id', 'dim', 'mode', 'llm_name'])['runtime_seconds'].agg(['mean', 'std', 'min', 'max']).reset_index()
        time_stats.columns = ['problem_id', 'dim', 'mode', 'llm_name', 'Avg Total Runtime (s)', 'Runtime Std', 'Min Runtime (s)', 'Max Runtime (s)']
        
        display(time_stats.round(2))
else:
    print("No database records available to calculate statistics.")

No database records available to calculate statistics.


## 3. Advanced Visualizations
Compare time consumed across different LLMs for the same problem, error distribution, and evolutionary convergence trajectories.

In [4]:
if not df_combined.empty:
    df_flat = df_combined.reset_index()
    
    # ---------------------------------------------------------
    # Plot 1: Average Runtime Comparison (Bar Chart)
    # Compare time consumed across different LLMs for the same problem
    # ---------------------------------------------------------
    runtime_per_exp = df_flat.groupby(['problem_id', 'llm_name', 'experiment_id'])['runtime_seconds'].sum().reset_index()
    avg_runtime = runtime_per_exp.groupby(['problem_id', 'llm_name'])['runtime_seconds'].mean().reset_index()
    
    # Ensure problem_id is treated as a categorical/string value for better x-axis spacing
    avg_runtime['problem_id_str'] = "Problem " + avg_runtime['problem_id'].astype(str)
    
    fig_runtime = px.bar(
        avg_runtime,
        x='problem_id_str',
        y='runtime_seconds',
        color='llm_name',
        barmode='group',
        title='Average Execution Time per Problem across LLMs',
        labels={'problem_id_str': 'BBOB Problem', 'runtime_seconds': 'Avg Total Runtime (s)', 'llm_name': 'LLM'}
    )
    fig_runtime.update_layout(template='plotly_white', height=500, width=900)
    fig_runtime.show()
    
    # ---------------------------------------------------------
    # Plot 2: Boxplot comparing final errors
    # ---------------------------------------------------------
    fig_box = px.box(
        df_exp,
        x='llm_name',
        y='best_final_error',
        color='mode',
        points='all',
        facet_col='problem_id',
        title='Best Final Error Distribution across LLMs (Faceted by Problem)',
        labels={'llm_name': 'LLM', 'best_final_error': 'Final Error', 'mode': 'Mode'}
    )
    # Log scale is often better for errors, but handle 0s or negatives carefully if they exist
    fig_box.update_layout(template='plotly_white', yaxis_type='log', height=500, width=1000)
    # Update all yaxes to log
    fig_box.update_yaxes(type="log")
    fig_box.show()
    
    # ---------------------------------------------------------
    # Plot 3: Convergence Trajectories
    # ---------------------------------------------------------
    fig_conv = go.Figure()
    
    # Limit to a few experiments if there are too many to prevent plot clutter
    for exp_id in df_flat['experiment_id'].unique()[:20]:
        exp_subset = df_flat[df_flat['experiment_id'] == exp_id].sort_values(by='iteration')
        meta_row = exp_subset.iloc[0]
        label = f"Prob {meta_row['problem_id']} | {meta_row['llm_name']} | Exp {exp_id}"
        
        fig_conv.add_trace(go.Scatter(
            x=exp_subset['iteration'],
            y=exp_subset['final_error'],
            mode='lines+markers',
            name=label,
            hovertext=exp_subset['algorithm_name']
        ))
        
    fig_conv.update_layout(
        title='Evolutionary Convergence Trajectories (First 20 Experiments)',
        xaxis_title='Iteration',
        yaxis_title='Final Error (Log Scale)',
        yaxis_type='log',
        template='plotly_white',
        height=600,
        width=1000
    )
    fig_conv.show()

else:
    print("No iteration data available for plotting.")

No iteration data available for plotting.
